# Reto 9 — Data Science: análisis de series temporales y volatilidad

En este cuaderno analizo las series de WMT, MRK y NVDA (precios ajustados diarios), estudio sus propiedades, modelizo su volatilidad con modelos ARCH/GARCH, evalúo la predicción fuera de muestra y obtengo la volatilidad y las correlaciones que necesita la simulación de Monte Carlo.

**Orden del análisis**
1. Validación del dataset y cálculo de retornos logarítmicos.
1b. Justificación de la cartera (criterios cuantitativos y cualitativos).
2. Análisis estructural: tendencia, estacionalidad y ruido.
3. Estacionariedad (ADF y KPSS) y 3 bis: detrending con splines y modelo estructural (componentes no observables).
4. Heterocedasticidad condicional (ARCH-LM, Ljung-Box, ACF/PACF).
5. Modelización de la volatilidad (ARCH y GARCH; comparación por AIC/BIC) y validación de mi implementación frente a `arch`.
6. Predicción de la volatilidad fuera de muestra.
7. Volatilidad y correlaciones para Monte Carlo.

**Antes de ejecutar**
- El CSV preprocesado debe estar en `Datos/dowjones_limpio.csv` con las columnas `date`, `symbol` y `adj_close`.
- Librerías: `numpy`, `pandas`, `scipy`, `statsmodels` y `plotly` (gráficos interactivos que además se exportan como HTML/JS a `figuras/`, tal y como piden los entregables). Los modelos GARCH están implementados por mí con `scipy` (máxima verosimilitud), sin librerías específicas de volatilidad.
- Ejecuto las celdas en orden. Las interpretaciones numéricas se generan a partir de los resultados de cada ejecución.

In [1]:
"""Reto 9 — Data Science: series temporales y volatilidad.

Dataset de entrada : Datos/dowjones_limpio.csv   (columnas: date, symbol, adj_close; ya preprocesado)
Salidas            : Datos/volatilidad_cartera.csv, Datos/correlaciones_cartera.csv
Cartera            : WMT, MRK, NVDA
"""
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, signal
from scipy.signal import lfilter, lfiltic
from scipy.special import gammaln
from scipy.optimize import minimize
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tools.numdiff import approx_hess
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tools.sm_exceptions import InterpolationWarning

warnings.filterwarnings("ignore", category=InterpolationWarning)   # KPSS: p-valor tabulado entre 0.01 y 0.1
warnings.filterwarnings("ignore", category=FutureWarning)

## 0. Configuración
`DT_PASO` es el paso de tiempo de la simulación de Monte Carlo (5 años entre 20 instantes = 0.25 años). La volatilidad que exporto está en años para que sea coherente con ese paso.

In [2]:
RUTA_DATOS = Path("Datos") / "dowjones_limpio.csv"
RUTA_SALIDA = Path("Datos") / "volatilidad_cartera.csv"
RUTA_RHO = Path("Datos") / "correlaciones_cartera.csv"
ACCIONES_CARTERA = ["WMT", "MRK", "NVDA"]

ALPHA_TEST = 0.05
DIAS_ANIO = 252
PERIODO_ESTACIONAL = 252      # hipótesis de ciclo anual, en sesiones
LAGS_DIAGNOSTICO = 10

N_TEST = 750                  # aprox. tres años de sesiones para evaluar fuera de muestra
REFIT_CADA = 20               # reestimo el GARCH cada 20 sesiones (ventana expansiva)

TTM_ANIOS, N_PASOS = 5, 20
DT_PASO = TTM_ANIOS / N_PASOS
FUENTE_SIGMA_MC = "garch_horizonte"   # "historica" | "garch_largo_plazo" | "garch_horizonte"

pd.set_option("display.max_columns", None); pd.set_option("display.width", 170)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
COLORES = {"WMT": "#1f77b4", "MRK": "#2ca02c", "NVDA": "#d62728"}

def si(cond, a, b): return a if cond else b

In [ ]:
# ---- Visualización: Plotly (HTML + JS interactivo) con la paleta corporativa BBVA ----
import itertools
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import pacf
from statsmodels.tsa.statespace.structural import UnobservedComponents
from scipy.interpolate import make_smoothing_spline

BBVA_COLORES = {
    "azul_principal":  "001391",
    "azul_claro":      "4D5FC4",
    "azul_muy_claro":  "B3BAE8",
    "gris_oscuro":     "1A1A2E",
    "gris_medio":      "6E7191",
    "gris_claro":      "E4E4EC",
    "blanco":          "FFFFFF",
    "rojo":            "FF4B4B",
}
BBVA = {k: "#" + v for k, v in BBVA_COLORES.items()}
COLORES = {"WMT": BBVA["azul_principal"], "MRK": BBVA["azul_claro"], "NVDA": BBVA["rojo"]}   # un color fijo por activo en todo el cuaderno

RUTA_GRAFICOS = Path("figuras"); RUTA_GRAFICOS.mkdir(exist_ok=True)
GRAFICOS = []

def mostrar(fig, nombre, titulo, alto=450):
    """Aplica el formato común, guarda el gráfico como HTML+JS autocontenido y lo muestra."""
    fig.update_layout(title=dict(text=f"<b>{titulo}</b>", font=dict(size=15, color=BBVA["azul_principal"])),
                      height=alto, template="simple_white", paper_bgcolor=BBVA["blanco"], plot_bgcolor=BBVA["blanco"],
                      font=dict(family="Arial", size=12, color=BBVA["gris_oscuro"]),
                      colorway=[BBVA["azul_principal"], BBVA["azul_claro"], BBVA["rojo"], BBVA["gris_medio"], BBVA["azul_muy_claro"]],
                      legend=dict(orientation="h", y=-0.15), margin=dict(l=60, r=30, t=70, b=50), hovermode="x unified")
    fig.update_xaxes(showgrid=True, gridcolor=BBVA["gris_claro"], linecolor=BBVA["gris_medio"])
    fig.update_yaxes(showgrid=True, gridcolor=BBVA["gris_claro"], linecolor=BBVA["gris_medio"])
    fig.write_html(RUTA_GRAFICOS / f"{nombre}.html", include_plotlyjs="cdn", full_html=True)
    if nombre not in GRAFICOS: GRAFICOS.append(nombre)
    fig.show()

def trazas_acf(x, nlags, color, parcial=False):
    """Barras de ACF/PACF con bandas de confianza al 95% (±1.96/√n)."""
    x = np.asarray(x, float)
    v = (pacf(x, nlags=nlags, method="ywm") if parcial else acf(x, nlags=nlags, fft=True))[1:]
    b = 1.96 / np.sqrt(len(x)); l = [1, nlags]
    banda = dict(color=BBVA["gris_medio"], dash="dot", width=1)
    return [go.Bar(x=np.arange(1, nlags + 1), y=v, marker_color=color, showlegend=False),
            go.Scatter(x=l, y=[b, b], mode="lines", line=banda, showlegend=False, hoverinfo="skip"),
            go.Scatter(x=l, y=[-b, -b], mode="lines", line=banda, showlegend=False, hoverinfo="skip")]

## 1. Carga, validación y retornos
El dataset ya viene preprocesado, así que aquí no repito la limpieza: compruebo el esquema, los tipos, los duplicados, que los precios sean positivos y que el calendario sea regular (sin huecos grandes entre sesiones). No relleno festivos ni imputo precios.

Trabajo con **retornos logarítmicos**, `r_t = ln(S_t / S_{t-1})`. Son los coherentes con el proceso browniano geométrico de la simulación, donde `S_t = S_{t-1}·exp(...)`, y son aditivos en el tiempo, lo que facilita pasar de volatilidad diaria a anual.

In [4]:
if not RUTA_DATOS.exists():
    raise FileNotFoundError(f"No se encuentra {RUTA_DATOS}.")
df = pd.read_csv(RUTA_DATOS)
faltan = {"date", "symbol", "adj_close"} - set(df.columns)
if faltan:
    raise ValueError(f"Faltan columnas: {sorted(faltan)}")

df = df[["date", "symbol", "adj_close"]].copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()
df["symbol"] = df["symbol"].astype("string").str.strip().str.upper()
df["adj_close"] = pd.to_numeric(df["adj_close"], errors="coerce")
if df.isna().any().any():
    raise ValueError("Hay fechas, símbolos o precios ausentes o no numéricos.")
if (df["adj_close"] <= 0).any():
    raise ValueError("Hay precios no positivos.")
if set(ACCIONES_CARTERA) - set(df["symbol"].unique()):
    raise ValueError("Faltan activos de la cartera en el CSV.")
if df.duplicated(["date", "symbol"]).any():
    raise ValueError("Hay más de un registro por fecha y símbolo.")

precios = (df[df["symbol"].isin(ACCIONES_CARTERA)]
           .pivot(index="date", columns="symbol", values="adj_close")
           .sort_index().reindex(columns=ACCIONES_CARTERA))
if precios.isna().any().any():
    raise ValueError("Fechas sin precio para algún activo:\n" + precios.isna().sum().to_string())

huecos = precios.index.to_series().diff().dt.days.dropna()
print("Días naturales entre sesiones consecutivas (3 = fin de semana, 4 = puente):")
print(huecos.value_counts().sort_index().to_string())
if (huecos > 4).any():
    print("\nAviso: hay huecos de más de 4 días, conviene revisar si faltan sesiones:")
    print(huecos[huecos > 4].to_string())

retornos = np.log(precios).diff().dropna()
rho = retornos.corr()

print("\n" + "=" * 78)
print(f"Periodo: {precios.index.min().date()} -> {precios.index.max().date()} | sesiones: {len(precios)}")
desc = retornos.describe().T
desc["asimetria"] = retornos.skew()
desc["curtosis_exceso"] = retornos.kurt()
desc["JB_pvalor"] = [stats.jarque_bera(retornos[a])[1] for a in ACCIONES_CARTERA]
print("\nDescriptivos de los retornos logarítmicos diarios:")
print(desc.round(6).to_string())
print("\nCorrelación entre retornos:"); print(rho.round(4))

fig = go.Figure()
for a in ACCIONES_CARTERA:
    fig.add_trace(go.Scatter(x=precios.index, y=precios[a] / precios[a].iloc[0], name=a, mode="lines", line=dict(color=COLORES[a], width=1.6)))
fig.update_yaxes(type="log", title="S_t / S_0 (escala log)")
mostrar(fig, "01_precios_normalizados", "Precio ajustado normalizado (base 1)", 430)

print("\nINTERPRETACIÓN")
vol_anual = retornos.std() * np.sqrt(DIAS_ANIO)
for a in ACCIONES_CARTERA:
    k, jb = desc.loc[a, "curtosis_exceso"], desc.loc[a, "JB_pvalor"]
    print(f"- {a}: volatilidad anualizada histórica {vol_anual[a]*100:.1f}%; curtosis en exceso {k:.1f} "
          f"y Jarque-Bera {si(jb < ALPHA_TEST, 'rechaza', 'no rechaza')} normalidad (p={jb:.3g}).")
off = rho.to_numpy()[np.triu_indices(3, 1)]
print(f"- Las correlaciones entre activos van de {off.min():.2f} a {off.max():.2f}: "
      f"{si(off.max() < 0.5, 'son bajas, la cartera diversifica bien', 'hay dependencia apreciable entre activos')}.")
print(f"- {vol_anual.idxmax()} es el activo más volátil ({vol_anual.max()/vol_anual.drop(vol_anual.idxmax()).mean():.1f} veces la media de los otros dos).")
if (desc["curtosis_exceso"] > 1).any():
    print("- Las colas son más pesadas que las de una normal, así que en la sección 5 pruebo también innovaciones t de Student.")

Días naturales entre sesiones consecutivas (3 = fin de semana, 4 = puente):
date
1.0    1485
2.0      23
3.0     346
4.0      49

Periodo: 2018-06-06 -> 2025-12-31 | sesiones: 1904

Descriptivos de los retornos logarítmicos diarios:
         count      mean       std       min       25%       50%       75%       max  asimetria  curtosis_exceso  JB_pvalor
symbol                                                                                                                     
WMT     1903.0  0.000783  0.013805 -0.120765 -0.005562  0.000808  0.007305  0.110723   0.058192        13.271708        0.0
MRK     1903.0  0.000429  0.014862 -0.103840 -0.006759  0.000581  0.008279  0.080421  -0.457509         6.990337        0.0
NVDA    1903.0  0.001759  0.032594 -0.207711 -0.015300  0.002835  0.019589  0.218088  -0.180419         4.755325        0.0

Correlación entre retornos:
symbol     WMT     MRK    NVDA
symbol                        
WMT     1.0000  0.2345  0.2255
MRK     0.2345  1.0000  0


INTERPRETACIÓN
- WMT: volatilidad anualizada histórica 21.9%; curtosis en exceso 13.3 y Jarque-Bera rechaza normalidad (p=0).
- MRK: volatilidad anualizada histórica 23.6%; curtosis en exceso 7.0 y Jarque-Bera rechaza normalidad (p=0).
- NVDA: volatilidad anualizada histórica 51.7%; curtosis en exceso 4.8 y Jarque-Bera rechaza normalidad (p=0).
- Las correlaciones entre activos van de 0.13 a 0.23: son bajas, la cartera diversifica bien.
- NVDA es el activo más volátil (2.3 veces la media de los otros dos).
- Las colas son más pesadas que las de una normal, así que en la sección 5 pruebo también innovaciones t de Student.


## 1b. Justificación de la cartera
La cartera (WMT, MRK, NVDA) se elige con criterios **cuantitativos** y **cualitativos**, calculados sobre todas las acciones del Dow Jones disponibles en el CSV:

- **Cuantitativos**: la cartera es el subyacente de una opción asiática sobre la *media geométrica* de tres activos, así que busco (i) **baja correlación** entre los activos (diversificación) y (ii) **perfiles de volatilidad distintos** (baja, media y alta), para que el payoff dependa de dinámicas distintas y el modelo basado en datos tenga un problema no trivial. Comparo el trío elegido con las C(N,3) combinaciones posibles.
- **Cualitativos**: un activo por sector con comportamiento económico distinto: consumo defensivo (WMT, ciclo estable), salud (MRK, poca relación con el ciclo tecnológico) y tecnología de alto crecimiento (NVDA, volátil y sensible a expectativas).

In [5]:
SECTOR = {"AAPL": "Tecnología", "AMGN": "Salud", "AMZN": "Consumo discrecional", "AXP": "Financiero", "BA": "Industrial",
          "CAT": "Industrial", "CRM": "Tecnología", "CSCO": "Tecnología", "CVX": "Energía", "DIS": "Comunicación",
          "DOW": "Materiales", "GS": "Financiero", "HD": "Consumo discrecional", "HON": "Industrial", "IBM": "Tecnología",
          "INTC": "Tecnología", "JNJ": "Salud", "JPM": "Financiero", "KO": "Consumo defensivo", "MCD": "Consumo discrecional",
          "MMM": "Industrial", "MRK": "Salud", "MSFT": "Tecnología", "NKE": "Consumo discrecional", "NVDA": "Tecnología",
          "PG": "Consumo defensivo", "SHW": "Materiales", "TRV": "Financiero", "UNH": "Salud", "V": "Financiero",
          "VZ": "Comunicación", "WBA": "Consumo defensivo", "WMT": "Consumo defensivo"}

precios_all = df.pivot(index="date", columns="symbol", values="adj_close").sort_index()
ret_all = np.log(precios_all).diff().iloc[1:]
resumen_all = pd.DataFrame({"vol_anual": ret_all.std() * np.sqrt(DIAS_ANIO),
                            "rend_anual": ret_all.mean() * DIAS_ANIO,
                            "sector": [SECTOR.get(s, "Otro") for s in ret_all.columns]}, index=ret_all.columns)
corr_all = ret_all.corr(min_periods=500)
sim = corr_all.to_numpy()
resumen_all["corr_media"] = (sim.sum(axis=1) - 1) / (len(sim) - 1)

# todas las combinaciones de 3 activos: correlación media por pares y nº de sectores distintos
simbolos = list(corr_all.columns); idx = {s: i for i, s in enumerate(simbolos)}
combos, cm, nsec = [], [], []
for t in itertools.combinations(simbolos, 3):
    i, j, k = (idx[s] for s in t)
    cm.append(np.nanmean([sim[i, j], sim[i, k], sim[j, k]])); combos.append(t)
    nsec.append(len({resumen_all.loc[s, "sector"] for s in t}))
comb = pd.DataFrame({"combo": combos, "corr_media": cm, "sectores": nsec})
mi_corr = float(np.mean([corr_all.loc["WMT", "MRK"], corr_all.loc["WMT", "NVDA"], corr_all.loc["MRK", "NVDA"]]))
pct_total = float((comb["corr_media"] < mi_corr).mean() * 100)
c3 = comb[comb["sectores"] == 3]
pct_3sec = float((c3["corr_media"] < mi_corr).mean() * 100)

print(resumen_all.loc[ACCIONES_CARTERA].round(3).to_string())
print(f"\nCorrelación media por pares de la cartera: {mi_corr:.3f}")
print(f"Combinaciones analizadas: {len(comb)} ({len(c3)} con tres sectores distintos)")

# Gráfico 1: riesgo-rentabilidad de todo el índice, cartera resaltada
fig = go.Figure()
otros = resumen_all.drop(index=ACCIONES_CARTERA)
fig.add_trace(go.Scatter(x=otros["vol_anual"] * 100, y=otros["rend_anual"] * 100, mode="markers+text", text=otros.index,
                         textposition="top center", name="Resto del Dow Jones",
                         marker=dict(color=BBVA["azul_muy_claro"], size=9), textfont=dict(color=BBVA["gris_medio"], size=10),
                         customdata=otros["sector"], hovertemplate="%{text} (%{customdata})<br>σ %{x:.1f}% · μ %{y:.1f}%<extra></extra>"))
for a in ACCIONES_CARTERA:
    fig.add_trace(go.Scatter(x=[resumen_all.loc[a, "vol_anual"] * 100], y=[resumen_all.loc[a, "rend_anual"] * 100], mode="markers+text",
                             text=[a], textposition="top center", name=f"{a} ({resumen_all.loc[a, 'sector']})",
                             marker=dict(color=COLORES[a], size=15, line=dict(color=BBVA["gris_oscuro"], width=1))))
fig.update_xaxes(title="Volatilidad anualizada (%)"); fig.update_yaxes(title="Rendimiento log anualizado (%)")
fig.update_layout(hovermode="closest")
mostrar(fig, "02_riesgo_rentabilidad_dowjones", "Riesgo-rentabilidad del Dow Jones y cartera elegida", 500)

# Gráfico 2: dónde queda la cartera frente a todas las combinaciones posibles
fig = go.Figure()
fig.add_trace(go.Histogram(x=comb["corr_media"], nbinsx=50, marker_color=BBVA["azul_muy_claro"], name="Todas las combinaciones"))
fig.add_trace(go.Histogram(x=c3["corr_media"], nbinsx=50, marker_color=BBVA["azul_claro"], name="Con 3 sectores distintos", opacity=0.8))
fig.add_vline(x=mi_corr, line=dict(color=BBVA["rojo"], width=2, dash="dash"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color=BBVA["rojo"], dash="dash"), name="WMT-MRK-NVDA"))
fig.update_layout(barmode="overlay", hovermode="closest"); fig.update_xaxes(title="Correlación media por pares"); fig.update_yaxes(title="Nº de combinaciones")
mostrar(fig, "03_correlacion_combinaciones", "Correlación media de la cartera frente a todas las combinaciones de 3 activos", 430)

print("\nINTERPRETACIÓN")
print(f"- La cartera tiene una correlación media de {mi_corr:.2f}; solo el {pct_total:.0f}% de las {len(comb)} combinaciones posibles es menos correlacionada "
      f"({pct_3sec:.0f}% si nos limitamos a las que cubren tres sectores).")
vols = resumen_all.loc[ACCIONES_CARTERA, "vol_anual"]
sect_txt = ", ".join(a + ": " + resumen_all.loc[a, "sector"] for a in ACCIONES_CARTERA)
print(f"- Cubre tres sectores ({sect_txt}) y un rango de volatilidad de {vols.min()*100:.0f}% a {vols.max()*100:.0f}%: perfil bajo, medio y alto.")
print("- Limitación: una correlación baja reduce la volatilidad de la media geométrica de la cesta y, con ella, el valor de la opción; la diversificación se elige por coherencia con la estrategia, no para maximizar la prima.")

        vol_anual  rend_anual             sector  corr_media
symbol                                                      
WMT         0.219       0.197  Consumo defensivo       0.292
MRK         0.236       0.108              Salud       0.306
NVDA        0.517       0.443         Tecnología       0.334

Correlación media por pares de la cartera: 0.196
Combinaciones analizadas: 4060 (2830 con tres sectores distintos)



INTERPRETACIÓN
- La cartera tiene una correlación media de 0.20; solo el 0% de las 4060 combinaciones posibles es menos correlacionada (0% si nos limitamos a las que cubren tres sectores).
- Cubre tres sectores (WMT: Consumo defensivo, MRK: Salud, NVDA: Tecnología) y un rango de volatilidad de 22% a 52%: perfil bajo, medio y alto.
- Limitación: una correlación baja reduce la volatilidad de la media geométrica de la cesta y, con ella, el valor de la opción; la diversificación se elige por coherencia con la estrategia, no para maximizar la prima.


## 2. Análisis estructural: tendencia, estacionalidad y ruido
Para cada serie sigo este esquema:
1. **ACF del log-precio**: un decaimiento muy lento indica tendencia (no estacionariedad en media).
2. **Periodograma de los retornos**: busca frecuencias dominantes; si hubiera estacionalidad anual aparecería un pico en el periodo 252.
3. **Descomposición aditiva del log-precio** con periodo 252 sesiones. El modelo aditivo sobre el logaritmo equivale a uno multiplicativo sobre el precio, que es lo razonable porque la amplitud de las variaciones crece con el nivel. La hago sobre el índice de sesión, sin crear cotizaciones artificiales en festivos.
4. **Contrastes sobre el residuo** (ADF, KPSS y ARCH). Además calculo la *fuerza estacional* `Fs = max(0, 1 − Var(residuo)/Var(estacional+residuo))`: si es cercana a 0, el componente estacional extraído no es relevante.

In [6]:
def fuerza_estacional(d):
    r = d.resid.dropna(); s = d.seasonal.loc[r.index]
    return float(max(0.0, 1 - r.var() / (s + r).var()))

filas_desc = []
for accion in ACCIONES_CARTERA:
    logp = np.log(precios[accion]); r = retornos[accion]

    f, P = signal.periodogram(r.to_numpy() - r.mean())
    per, pot = 1 / f[1:], P[1:]
    top = per[np.argsort(pot)[::-1][:3]]
    acf_lp = trazas_acf(logp, 300, COLORES[accion])
    fig = make_subplots(rows=1, cols=3, subplot_titles=(f"Log-precio, {accion}", "ACF del log-precio", "Periodograma de retornos (líneas: 5, 21, 63, 252)"))
    fig.add_trace(go.Scatter(x=logp.index, y=logp.to_numpy(), mode="lines", line=dict(color=COLORES[accion], width=1.4), showlegend=False), row=1, col=1)
    for t_ in acf_lp: fig.add_trace(t_, row=1, col=2)
    fig.add_trace(go.Scatter(x=per, y=pot, mode="lines", line=dict(color=COLORES[accion], width=1), showlegend=False), row=1, col=3)
    for p_ in (5, 21, 63, 252):
        fig.add_trace(go.Scatter(x=[p_, p_], y=[pot.min(), pot.max()], mode="lines", line=dict(color=BBVA["gris_medio"], dash="dot", width=1),
                                 showlegend=False, hoverinfo="skip"), row=1, col=3)
    fig.update_xaxes(type="log", title="Periodo (sesiones)", row=1, col=3); fig.update_yaxes(type="log", row=1, col=3)
    fig.update_xaxes(title="Retardo", row=1, col=2); fig.update_layout(hovermode="closest")
    mostrar(fig, f"04_{accion}_acf_periodograma", f"{accion}: tendencia, memoria y periodicidad", 380)

    sesiones = pd.Series(logp.to_numpy(), index=pd.RangeIndex(len(logp), name="sesion"))

    sesiones = pd.Series(logp.to_numpy(), index=pd.RangeIndex(len(logp), name="sesion"))
    d = seasonal_decompose(sesiones, model="additive", period=PERIODO_ESTACIONAL)
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                        subplot_titles=("Observado (log-precio)", "Tendencia", f"Estacional (periodo {PERIODO_ESTACIONAL})", "Residuo"))
    for k_, (comp_, col_) in enumerate(((d.observed, COLORES[accion]), (d.trend, BBVA["azul_principal"]), (d.seasonal, BBVA["azul_claro"]), (d.resid, BBVA["gris_medio"])), 1):
        fig.add_trace(go.Scatter(x=comp_.index, y=comp_.to_numpy(), mode="lines", line=dict(color=col_, width=1.2), showlegend=False), row=k_, col=1)
    fig.update_xaxes(title="Sesión", row=4, col=1)
    mostrar(fig, f"05_{accion}_descomposicion", f"Descomposición aditiva del log-precio, {accion}", 640)

    res = d.resid.dropna()
    filas_desc.append({
        "accion": accion,
        "rango_tendencia": float(d.trend.max() - d.trend.min()),
        "amplitud_estacional": float(d.seasonal.max() - d.seasonal.min()),
        "fuerza_estacional": fuerza_estacional(d),
        "periodos_dominantes": ", ".join(f"{x:.0f}" for x in top),
        "pico_cerca_252": bool(any(abs(x - 252) / 252 < 0.1 for x in top)),
        "resid_ADF_p": adfuller(res, autolag="AIC")[1],
        "resid_KPSS_p": kpss(res, regression="c", nlags="auto")[1],
        "resid_ARCH_p": het_arch(res.to_numpy(), nlags=LAGS_DIAGNOSTICO)[1],
    })
tabla_descomposicion = pd.DataFrame(filas_desc).set_index("accion")
print(tabla_descomposicion.to_string(float_format=lambda x: f"{x:.4g}"))

print("\nINTERPRETACIÓN")
for a in ACCIONES_CARTERA:
    t = tabla_descomposicion.loc[a]
    rel = t["amplitud_estacional"] / t["rango_tendencia"]
    print(f"- {a}: la tendencia recorre {t['rango_tendencia']:.2f} unidades de log-precio frente a una amplitud estacional de "
          f"{t['amplitud_estacional']:.2f} ({rel*100:.0f}% de la tendencia); fuerza estacional {t['fuerza_estacional']:.2f}. "
          + si(rel < 0.2 and not t["pico_cerca_252"],
               "No veo estacionalidad anual relevante: el componente estacional es pequeño frente a la tendencia y el periodograma no tiene un pico en 252 (periodos dominantes: " + t["periodos_dominantes"] + ").",
               si(t["pico_cerca_252"],
                  "El periodograma muestra un pico cerca de 252 sesiones, así que hay indicio de estacionalidad anual que conviene confirmar.",
                  "El componente estacional extraído no es despreciable frente a la tendencia, aunque el periodograma no confirma un ciclo anual; lo interpreto con cautela.")))
    print(f"  El residuo {si(t['resid_ADF_p'] < ALPHA_TEST and t['resid_KPSS_p'] >= ALPHA_TEST, 'es estacionario en media', 'no queda claramente estacionario en media')} "
          f"(ADF p={t['resid_ADF_p']:.3g}, KPSS p≥{t['resid_KPSS_p']:.2g}) y "
          f"{si(t['resid_ARCH_p'] < ALPHA_TEST, 'presenta efectos ARCH', 'no presenta efectos ARCH')} (p={t['resid_ARCH_p']:.3g}).")
print("La tendencia del log-precio es la que hace no estacionario el precio, por eso modelizo los retornos y no los precios.")

        rango_tendencia  amplitud_estacional  fuerza_estacional periodos_dominantes  pico_cerca_252  resid_ADF_p  resid_KPSS_p  resid_ARCH_p
accion                                                                                                                                      
WMT               1.235              0.09591             0.1925             5, 2, 3           False    2.348e-07           0.1    2.249e-291
MRK              0.6959              0.07497            0.07263             2, 2, 2           False     3.34e-05           0.1    4.443e-279
NVDA              3.607               0.1719            0.08201             2, 2, 2           False    5.341e-05           0.1    9.619e-301

INTERPRETACIÓN
- WMT: la tendencia recorre 1.24 unidades de log-precio frente a una amplitud estacional de 0.10 (8% de la tendencia); fuerza estacional 0.19. No veo estacionalidad anual relevante: el componente estacional es pequeño frente a la tendencia y el periodograma no tiene un pico en 2

## 3. Estacionariedad: ADF y KPSS
- **ADF**: H0 = la serie tiene raíz unitaria (no estacionaria). Rechazo si p < 0.05.
- **KPSS**: H0 = la serie es estacionaria. Rechazo si p < 0.05.

Como las hipótesis nulas son contrarias, los interpreto conjuntamente. Contrasto el **log-precio** con tendencia determinista (`ct`) y el **retorno** con constante (`c`). Lo esperable en un activo financiero es que el precio tenga raíz unitaria y que una diferencia (el retorno) sea estacionaria.

In [7]:
def contrasta(serie, reg):
    serie = pd.Series(serie).astype(float).dropna()
    adf_s, adf_p = adfuller(serie, regression=reg, autolag="AIC")[:2]
    kp_s, kp_p = kpss(serie, regression=reg, nlags="auto")[:2]
    if adf_p < ALPHA_TEST and kp_p >= ALPHA_TEST: c = "Estacionaria"
    elif adf_p >= ALPHA_TEST and kp_p < ALPHA_TEST: c = "No estacionaria"
    else: c = "No concluyente"
    return {"ADF_stat": adf_s, "ADF_p": adf_p, "KPSS_stat": kp_s, "KPSS_p": kp_p, "conclusion": c}

filas = []
for a in ACCIONES_CARTERA:
    filas.append({"accion": a, "serie": "log-precio (ct)", **contrasta(np.log(precios[a]), "ct")})
    filas.append({"accion": a, "serie": "retorno log (c)", **contrasta(retornos[a], "c")})
tabla_estacionariedad = pd.DataFrame(filas).set_index(["accion", "serie"])
print(tabla_estacionariedad.to_string(float_format=lambda x: f"{x:.4g}"))
print("Nota: si KPSS da 0.1, es el límite de la tabla (el p-valor real es mayor).")

print("\nINTERPRETACIÓN")
for a in ACCIONES_CARTERA:
    cp = tabla_estacionariedad.loc[(a, "log-precio (ct)"), "conclusion"]
    cr = tabla_estacionariedad.loc[(a, "retorno log (c)"), "conclusion"]
    txt = f"- {a}: log-precio → {cp.lower()}; retorno → {cr.lower()}."
    if cp != "Estacionaria" and cr == "Estacionaria":
        txt += " El precio es integrado de orden 1 y una diferencia basta para estacionarlo."
    print(txt)
print("Aquí 'estacionaria' significa estacionaria en media (sin raíz unitaria). No contradice que la varianza condicional cambie "
      "en el tiempo, que es lo que estudio en la sección siguiente.")

                        ADF_stat     ADF_p  KPSS_stat  KPSS_p       conclusion
accion serie                                                                  
WMT    log-precio (ct)    -1.733     0.736      1.019    0.01  No estacionaria
       retorno log (c)    -14.77 2.321e-27     0.1049     0.1     Estacionaria
MRK    log-precio (ct)    -2.418    0.3701     0.5646    0.01  No estacionaria
       retorno log (c)    -14.64 3.614e-27    0.07498     0.1     Estacionaria
NVDA   log-precio (ct)    -2.686    0.2419     0.4706    0.01  No estacionaria
       retorno log (c)    -14.01 3.661e-26       0.19     0.1     Estacionaria
Nota: si KPSS da 0.1, es el límite de la tabla (el p-valor real es mayor).

INTERPRETACIÓN
- WMT: log-precio → no estacionaria; retorno → estacionaria. El precio es integrado de orden 1 y una diferencia basta para estacionarlo.
- MRK: log-precio → no estacionaria; retorno → estacionaria. El precio es integrado de orden 1 y una diferencia basta para estacionarlo.
- N

## 3 bis. Detrending con splines y modelo estructural (componentes no observables)
Complemento la descomposición clásica con dos técnicas más flexibles, para decidir con evidencia cómo procesar las series:

1. **Detrending con spline de suavizado** (penalización elegida por validación cruzada generalizada): se resta una tendencia suave y se contrasta si el residuo es estacionario.
2. **Modelo estructural** (`UnobservedComponents`): nivel + pendiente estocásticos (*local linear trend*) y estacionalidad semanal de 5 sesiones, estimado por máxima verosimilitud con filtro de Kalman (con más iteraciones del optimizador y partiendo de la solución de otro algoritmo, para que el resultado no dependa del punto de partida). Informa de cuánta varianza se atribuye a cada componente. Uso 5 sesiones porque es la única periodicidad con base económica en datos diarios y el periodograma la señala; un ciclo de 252 sesiones solo da unos 7 ciclos y no aparece en el periodograma.

Si un spline flexible deja un residuo estacionario eso **no** prueba que la tendencia sea determinista: un spline muy flexible puede absorber parte de una raíz unitaria. Por eso la decisión final se apoya también en el contraste de la sección 3 y en la diferenciación (retornos).

In [8]:
filas_uc = []
for accion in ACCIONES_CARTERA:
    logp = np.log(precios[accion]); y = logp.to_numpy(); x = np.arange(len(y), dtype=float)

    spl = make_smoothing_spline(x, y)              # lambda por GCV
    tend_sp = spl(x); det_sp = y - tend_sp

    mod_uc = UnobservedComponents(y, level="local linear trend", seasonal=5)
    ini_uc = mod_uc.fit(disp=False, method="powell", maxiter=5000)        # otro algoritmo para obtener un buen punto de partida
    uc = mod_uc.fit(start_params=ini_uc.params, disp=False, maxiter=2000)  # refinamiento desde ese punto y con más iteraciones
    nivel, est = uc.smoothed_state[0], uc.smoothed_state[2]
    irreg = y - nivel - est
    par = dict(zip(uc.param_names, uc.params))

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=("Log-precio, tendencia spline y nivel estructural", "Componente estacional semanal (5 sesiones)", "Irregular (ruido)"))
    fig.add_trace(go.Scatter(x=logp.index, y=y, name="Log-precio", mode="lines", line=dict(color=BBVA["azul_muy_claro"], width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=logp.index, y=tend_sp, name="Spline", mode="lines", line=dict(color=BBVA["azul_principal"], width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=logp.index, y=nivel, name="Nivel estructural", mode="lines", line=dict(color=BBVA["rojo"], width=1.4, dash="dot")), row=1, col=1)
    fig.add_trace(go.Scatter(x=logp.index[-250:], y=est[-250:], name="Estacional", mode="lines", line=dict(color=BBVA["azul_claro"], width=1.4), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=logp.index, y=irreg, name="Irregular", mode="lines", line=dict(color=BBVA["gris_medio"], width=0.9), showlegend=False), row=3, col=1)
    mostrar(fig, f"06_{accion}_spline_estructural", f"{accion}: spline y modelo estructural", 680)

    c_sp = contrasta(det_sp, "c")
    filas_uc.append({"accion": accion, "var_nivel": par.get("sigma2.level", np.nan), "var_pendiente": par.get("sigma2.trend", np.nan),
                     "var_estacional": par.get("sigma2.seasonal", np.nan), "var_irregular": par.get("sigma2.irregular", np.nan),
                     "amplitud_estacional_semanal": float(est.max() - est.min()),
                     "spline_ADF_p": c_sp["ADF_p"], "spline_KPSS_p": c_sp["KPSS_p"], "spline_residuo": c_sp["conclusion"]})
tabla_estructural = pd.DataFrame(filas_uc).set_index("accion")
print(tabla_estructural.to_string(float_format=lambda x: f"{x:.4g}"))

print("\nINTERPRETACIÓN")
for a in ACCIONES_CARTERA:
    t = tabla_estructural.loc[a]
    tot = t[["var_nivel", "var_pendiente", "var_estacional", "var_irregular"]].fillna(0)
    dom = tot.idxmax().replace("var_", "")
    print(f"- {a}: la varianza del modelo estructural se concentra en el componente '{dom}'; amplitud estacional semanal {t['amplitud_estacional_semanal']:.4f} "
          f"en log-precio ({si(t['amplitud_estacional_semanal'] < 0.02, 'despreciable', 'a vigilar')}). "
          f"Residuo del spline: {t['spline_residuo'].lower()} (ADF p={t['spline_ADF_p']:.3g}, KPSS p≥{t['spline_KPSS_p']:.2g}).")
hay_est = (tabla_estructural["amplitud_estacional_semanal"] >= 0.02).any()
print("Conclusión de procesamiento: el nivel del modelo estructural cambia con el tiempo (tendencia estocástica) y "
      + ("hay algún componente estacional semanal a vigilar; " if hay_est else "no hay estacionalidad relevante; ")
      + "diferenciar (retornos logarítmicos) es el tratamiento más seguro y es el que alimenta ARCH/GARCH.")

        var_nivel  var_pendiente  var_estacional  var_irregular  amplitud_estacional_semanal  spline_ADF_p  spline_KPSS_p spline_residuo
accion                                                                                                                                  
WMT     0.0001713      3.428e-16       2.651e-14      9.709e-06                     0.001149             0            0.1   Estacionaria
MRK     0.0001867      2.391e-14       7.621e-10      1.701e-05                     0.001113             0            0.1   Estacionaria
NVDA    0.0009057      1.349e-09       7.828e-13      7.695e-05                     0.002585             0            0.1   Estacionaria

INTERPRETACIÓN
- WMT: la varianza del modelo estructural se concentra en el componente 'nivel'; amplitud estacional semanal 0.0011 en log-precio (despreciable). Residuo del spline: estacionaria (ADF p=0, KPSS p≥0.1).
- MRK: la varianza del modelo estructural se concentra en el componente 'nivel'; amplitud estacio

## 4. Heterocedasticidad condicional y autocorrelación
Sobre el retorno aplico:
- **ARCH-LM** (H0: no hay efectos ARCH).
- **Ljung-Box sobre `r`** (H0: sin autocorrelación en la media), para decidir si necesito un modelo ARMA para la media.
- **Ljung-Box sobre `r²`** (H0: sin dependencia en la varianza).

Dibujo la ACF y PACF de `r` y de `r²`: las de `r` orientan sobre la media y las de `r²` sobre los órdenes (p, q) del modelo de varianza.

In [9]:
filas = []
for a in ACCIONES_CARTERA:
    r = retornos[a] * 100
    lm, lm_p = het_arch(r.to_numpy(), nlags=LAGS_DIAGNOSTICO)[:2]
    lb_r = acorr_ljungbox(r, lags=[LAGS_DIAGNOSTICO], return_df=True)["lb_pvalue"].iloc[0]
    lb_r2 = acorr_ljungbox(r**2, lags=[LAGS_DIAGNOSTICO], return_df=True)["lb_pvalue"].iloc[0]
    filas.append({"accion": a, "ARCH_LM": lm, "ARCH_p": lm_p, "LjungBox_r_p": lb_r, "LjungBox_r2_p": lb_r2,
                  "max_abs_acf_r(10)": float(np.max(np.abs(acf(r, nlags=10)[1:]))),
                  "max_abs_acf_r2(10)": float(np.max(np.abs(acf(r**2, nlags=10)[1:])))})
tabla_arch = pd.DataFrame(filas).set_index("accion")
print(tabla_arch.to_string(float_format=lambda x: f"{x:.4g}"))

fig = make_subplots(rows=3, cols=4, vertical_spacing=0.09, horizontal_spacing=0.05,
                    subplot_titles=[f"{a}: {t_}" for a in ACCIONES_CARTERA for t_ in ("ACF de r", "PACF de r", "ACF de r²", "PACF de r²")])
for i, a in enumerate(ACCIONES_CARTERA, 1):
    r = retornos[a] * 100
    for j, (serie, parcial) in enumerate(((r, False), (r, True), (r**2, False), (r**2, True)), 1):
        for t_ in trazas_acf(serie, 30, COLORES[a], parcial): fig.add_trace(t_, row=i, col=j)
fig.update_layout(hovermode="closest")
mostrar(fig, "07_acf_pacf_retornos", "Autocorrelación de los retornos (media) y de sus cuadrados (varianza)", 760)

print("\nINTERPRETACIÓN")
for a in ACCIONES_CARTERA:
    t = tabla_arch.loc[a]
    print(f"- {a}: ARCH-LM {si(t['ARCH_p'] < ALPHA_TEST, 'rechaza', 'no rechaza')} la ausencia de efectos ARCH (p={t['ARCH_p']:.3g}); "
          f"Ljung-Box sobre r² p={t['LjungBox_r2_p']:.3g}, sobre r p={t['LjungBox_r_p']:.3g}.")
    if t["LjungBox_r_p"] < ALPHA_TEST:
        print(f"  Hay algo de autocorrelación en la media, pero la mayor autocorrelación de r es solo {t['max_abs_acf_r(10)']:.3f}; "
              "la considero económicamente pequeña y mantengo una media constante (lo anoto como limitación).")
    else:
        print("  No hay autocorrelación en la media, así que una constante es suficiente para modelarla.")
if (tabla_arch["ARCH_p"] < ALPHA_TEST).all():
    print("En los tres activos la varianza depende del pasado: no es constante, y un modelo ARCH/GARCH está justificado.")

        ARCH_LM    ARCH_p  LjungBox_r_p  LjungBox_r2_p  max_abs_acf_r(10)  max_abs_acf_r2(10)
accion                                                                                       
WMT       211.2  7.33e-40     0.0003519      1.411e-65            0.06878               0.266
MRK       153.9 5.785e-28     4.586e-10      1.744e-57            0.07899              0.1704
NVDA      91.81 2.341e-15     5.007e-05       3.27e-27            0.07858              0.1594



INTERPRETACIÓN
- WMT: ARCH-LM rechaza la ausencia de efectos ARCH (p=7.33e-40); Ljung-Box sobre r² p=1.41e-65, sobre r p=0.000352.
  Hay algo de autocorrelación en la media, pero la mayor autocorrelación de r es solo 0.069; la considero económicamente pequeña y mantengo una media constante (lo anoto como limitación).
- MRK: ARCH-LM rechaza la ausencia de efectos ARCH (p=5.78e-28); Ljung-Box sobre r² p=1.74e-57, sobre r p=4.59e-10.
  Hay algo de autocorrelación en la media, pero la mayor autocorrelación de r es solo 0.079; la considero económicamente pequeña y mantengo una media constante (lo anoto como limitación).
- NVDA: ARCH-LM rechaza la ausencia de efectos ARCH (p=2.34e-15); Ljung-Box sobre r² p=3.27e-27, sobre r p=5.01e-05.
  Hay algo de autocorrelación en la media, pero la mayor autocorrelación de r es solo 0.079; la considero económicamente pequeña y mantengo una media constante (lo anoto como limitación).
En los tres activos la varianza depende del pasado: no es constante, y 

## 5. Modelización de la volatilidad: ARCH y GARCH
Estimo los modelos por **máxima verosimilitud** con mi propia implementación en `scipy`. La media es una constante y el residuo es `ε_t = σ_t·z_t`, con

`σ²_t = ω + Σ α_i ε²_{t-i} + Σ β_j σ²_{t-j}`

- La recursión de la varianza la calculo con `lfilter` partiendo de la varianza muestral como valor inicial.
- Parametrizo (ω > 0, α, β ≥ 0, Σα + Σβ < 1) con transformaciones para que el optimizador trabaje sin restricciones (BFGS y refinamiento con Nelder-Mead desde varios puntos iniciales).
- Los errores estándar salen de la Hessiana numérica con corrección robusta (sándwich), apropiada cuando las colas son pesadas.
- Las innovaciones pueden ser normales o t de Student estandarizadas (parámetro ν).

Comparo ARCH(1), ARCH(5), GARCH(1,1), GARCH(2,1) y GARCH(1,2) por log-verosimilitud, AIC y BIC (mejor = menor). Antes de usar la implementación la contrasté con la librería `arch` sobre datos simulados con parámetros conocidos y coincide en parámetros, verosimilitud y errores estándar.

In [10]:
def varianza_condicional(eps, omega, alphas, betas, v0):
    alphas = np.asarray(alphas, float); betas = np.asarray(betas, float)
    p, q = len(alphas), len(betas)
    b = np.r_[0.0, alphas]; a = np.r_[1.0, -betas]
    m = omega / (1.0 - betas.sum())
    zi = lfiltic(b, a, y=np.full(q, v0 - m), x=np.full(p, v0))
    return m + lfilter(b, a, eps**2, zi=zi)[0]

def ll_obs(y, mu, omega, alphas, betas, nu=None, v0=None):
    eps = y - mu
    s2 = varianza_condicional(eps, omega, alphas, betas, np.var(y) if v0 is None else v0)
    if not np.all(np.isfinite(s2)) or np.any(s2 <= 0):
        return np.full(len(y), -np.inf)
    if nu is None:
        return -0.5 * (np.log(2*np.pi) + np.log(s2) + eps**2 / s2)
    c = gammaln((nu+1)/2) - gammaln(nu/2) - 0.5*np.log(np.pi*(nu-2))
    return c - 0.5*np.log(s2) - 0.5*(nu+1)*np.log1p(eps**2 / (s2*(nu-2)))

def log_verosimilitud(y, mu, omega, alphas, betas, nu=None):
    return float(np.sum(ll_obs(y, mu, omega, alphas, betas, nu)))

def _natural(u, p, q, t):
    omega = np.exp(u[1]); P = 0.9999 / (1 + np.exp(-u[2])); k = p + q
    l = np.r_[0.0, u[3:3+k-1]]; s = np.exp(l - l.max()); s /= s.sum()
    nu = 2.1 + np.exp(u[3+k-1]) if t else None
    return u[0], omega, P*s[:p], P*s[p:], nu

def _libre(mu, omega, alphas, betas, nu, p, q):
    w = np.r_[alphas, betas]; P = w.sum() / 0.9999; P = min(max(P, 1e-4), 1-1e-4)
    s = np.clip(w / w.sum(), 1e-6, None); l = np.log(s) - np.log(s[0])
    u = [mu, np.log(omega), np.log(P/(1-P))] + list(l[1:])
    if nu is not None: u.append(np.log(max(nu-2.1, 1e-3)))
    return np.array(u, float)

def ajustar(y, p=1, q=1, dist="normal", inicio=None):
    """Máxima verosimilitud de ARCH(p) (q=0) o GARCH(p,q) con innovaciones normal o t estandarizada."""
    y = np.asarray(y, float); t = dist == "t"; v = np.var(y); mu0 = y.mean()
    def nll(u):
        with np.errstate(all="ignore"):
            mu, om, al, be, nu = _natural(u, p, q, t)
            ll = log_verosimilitud(y, mu, om, al, be, nu)
        return -ll if np.isfinite(ll) else 1e12
    starts = []
    for P in (0.5, 0.8, 0.95):
        if q == 0: al = np.full(p, P/p); be = np.array([])
        else:
            al = np.full(p, 0.1*P/p); be = np.full(q, 0.9*P/q)
        starts.append(_libre(mu0, v*(1-P), al, be, 8.0 if t else None, p, q))
    if inicio is not None: starts.append(inicio)
    mejor = None
    for u0 in starts:
        r = minimize(nll, u0, method="BFGS", options={"maxiter": 300, "gtol": 1e-6})
        r = minimize(nll, r.x, method="Nelder-Mead",
                     options={"xatol": 1e-7, "fatol": 1e-9, "maxiter": 3000, "maxfev": 6000})
        if mejor is None or r.fun < mejor.fun: mejor = r
    mu, om, al, be, nu = _natural(mejor.x, p, q, t)
    theta = np.r_[mu, om, al, be] if not t else np.r_[mu, om, al, be, nu]
    nombres = ["mu", "omega"] + [f"alpha[{i+1}]" for i in range(p)] + [f"beta[{j+1}]" for j in range(q)] + (["nu"] if t else [])
    def nll_nat(th):
        mu_, om_, al_, be_ = th[0], th[1], th[2:2+p], th[2+p:2+p+q]
        nu_ = th[-1] if t else None
        if om_ <= 0 or np.any(al_ < 0) or np.any(be_ < 0) or al_.sum()+be_.sum() >= 1 or (t and nu_ <= 2.05):
            return 1e12
        return -log_verosimilitud(y, mu_, om_, al_, be_, nu_)
    v_y = np.var(y)
    def ll_vec(th):
        nu_ = th[-1] if t else None
        return ll_obs(y, th[0], th[1], th[2:2+p], th[2+p:2+p+q], nu_, v_y)
    try:
        H = approx_hess(theta, nll_nat)                       # Hessiana de -logL
        h = 1e-5 * np.maximum(np.abs(theta), 1e-2)
        S = np.empty((len(y), len(theta)))                    # scores por observación (dif. centradas)
        for j in range(len(theta)):
            d = np.zeros(len(theta)); d[j] = h[j]
            S[:, j] = (ll_vec(theta + d) - ll_vec(theta - d)) / (2*h[j])
        Hi = np.linalg.inv(H)
        cov = Hi @ (S.T @ S) @ Hi                             # sándwich (Bollerslev-Wooldridge)
        se = np.sqrt(np.diag(cov))
        se[theta < 1e-6] = np.nan                             # parámetro en la frontera: SE no válido
        if not np.all(np.isfinite(se[theta >= 1e-6])): raise np.linalg.LinAlgError
    except Exception:
        se = np.full(len(theta), np.nan)
    k = len(theta); ll = -float(mejor.fun)
    eps = y - mu; s2 = varianza_condicional(eps, om, al, be, v)
    tv = theta / se
    pv = 2 * (1 - stats.norm.cdf(np.abs(tv)))
    return {"p": p, "q": q, "dist": dist, "nombres": nombres, "theta": theta, "se": se, "t": tv, "pvalor": pv,
            "loglik": ll, "aic": 2*k - 2*ll, "bic": np.log(len(y))*k - 2*ll, "sigma2": s2,
            "resid_std": eps/np.sqrt(s2), "u": mejor.x, "n_par": k,
            "convergencia": bool(np.isfinite(mejor.fun) and mejor.fun < 1e11)}

In [11]:
def especificar(nombre):
    base, dist = nombre.rsplit("-", 1)
    dist = {"N": "normal", "t": "t"}[dist]
    if base.startswith("ARCH"):
        return dict(p=int(base[5:-1]), q=0, dist=dist)
    p, q = base[6:-1].split(",")
    return dict(p=int(p), q=int(q), dist=dist)

CANDIDATOS = ["ARCH(1)-N", "ARCH(5)-N", "GARCH(1,1)-N", "GARCH(1,1)-t", "GARCH(2,1)-t", "GARCH(1,2)-t"]

seleccion, modelos_final, spec_final = [], {}, {}
for a in ACCIONES_CARTERA:
    y = retornos[a].to_numpy() * 100
    ajustes = {c: ajustar(y, **especificar(c)) for c in CANDIDATOS}
    for c, m in ajustes.items():
        seleccion.append({"accion": a, "modelo": c, "loglik": m["loglik"], "AIC": m["aic"], "BIC": m["bic"],
                          "convergencia": m["convergencia"]})
    g = {c: ajustes[c]["bic"] for c in ("GARCH(1,1)-N", "GARCH(1,1)-t")}
    spec_final[a] = min(g, key=g.get)          # GARCH(1,1) con la distribución de menor BIC
    modelos_final[a] = ajustes[spec_final[a]]

tabla_seleccion = pd.DataFrame(seleccion).set_index(["accion", "modelo"])
print("Comparación de modelos (menor AIC/BIC = mejor):")
print(tabla_seleccion.to_string(float_format=lambda x: f"{x:.2f}"))
mejor_bic = tabla_seleccion.groupby("accion")["BIC"].idxmin().map(lambda t: t[1])
print("\nMejor BIC global:", mejor_bic.to_dict())
print("Especificación final (GARCH(1,1), distribución elegida por BIC):", spec_final)

Comparación de modelos (menor AIC/BIC = mejor):
                      loglik     AIC     BIC  convergencia
accion modelo                                             
WMT    ARCH(1)-N    -3209.59 6425.18 6441.84          True
       ARCH(5)-N    -3165.53 6345.05 6383.91          True
       GARCH(1,1)-N -3166.05 6340.10 6362.30          True
       GARCH(1,1)-t -2917.41 5844.83 5872.58          True
       GARCH(2,1)-t -2917.41 5846.83 5880.13          True
       GARCH(1,2)-t -2916.28 5844.56 5877.87          True
MRK    ARCH(1)-N    -3400.25 6806.50 6823.15          True
       ARCH(5)-N    -3360.01 6734.02 6772.88          True
       GARCH(1,1)-N -3346.45 6700.91 6723.11          True
       GARCH(1,1)-t -3192.26 6394.51 6422.27          True
       GARCH(2,1)-t -3192.26 6396.51 6429.82          True
       GARCH(1,2)-t -3191.85 6395.69 6429.00          True
NVDA   ARCH(1)-N    -4912.77 9831.54 9848.19          True
       ARCH(5)-N    -4834.61 9683.21 9722.07          True
       G

Elijo **GARCH(1,1)** como especificación final porque es el estándar en finanzas, es parsimonioso y su varianza de largo plazo `ω/(1−α−β)` tiene expresión cerrada, que es lo que necesito para Monte Carlo. La distribución (normal o t) la decido por BIC. Compruebo después que los coeficientes son significativos y que los residuos estandarizados no conservan autocorrelación.

In [12]:
filas_par, filas_diag = [], []
fig = make_subplots(rows=len(ACCIONES_CARTERA), cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=[f"{a} [{spec_final[a]}]" for a in ACCIONES_CARTERA])
for i_, a in enumerate(ACCIONES_CARTERA, 1):
    m = modelos_final[a]; d = dict(zip(m["nombres"], m["theta"]))
    filas_par.append(pd.DataFrame({"accion": a, "parametro": m["nombres"], "coef": m["theta"], "std_err": m["se"],
                                   "t": m["t"], "p_valor": m["pvalor"]}))
    om, al, be = d["omega"], d["alpha[1]"], d["beta[1]"]
    z = pd.Series(m["resid_std"])
    lb1 = acorr_ljungbox(z, lags=[LAGS_DIAGNOSTICO], return_df=True)["lb_pvalue"].iloc[0]
    lb2 = acorr_ljungbox(z**2, lags=[LAGS_DIAGNOSTICO], return_df=True)["lb_pvalue"].iloc[0]
    pers = al + be
    sig_ok = [pv < ALPHA_TEST for n_, pv in zip(m["nombres"], m["pvalor"]) if n_ not in ("mu",) and np.isfinite(pv)]
    filas_diag.append({"accion": a, "alpha": al, "beta": be, "alpha+beta": pers,
                       "semivida_dias": np.log(0.5) / np.log(pers), "vol_LP_diaria_%": np.sqrt(om / (1 - pers)),
                       "nu": d.get("nu", np.nan), "LB_resid_p": lb1, "LB_resid2_p": lb2,
                       "coefs_significativos": bool(all(sig_ok))})
    fig.add_trace(go.Scatter(x=retornos.index, y=np.sqrt(m["sigma2"]) * np.sqrt(DIAS_ANIO), name=f"{a}: σ condicional", mode="lines",
                             line=dict(color=COLORES[a], width=1.2), showlegend=False), row=i_, col=1)
    lp_ = np.sqrt(om / (1 - pers)) * np.sqrt(DIAS_ANIO)
    fig.add_trace(go.Scatter(x=[retornos.index[0], retornos.index[-1]], y=[lp_, lp_], mode="lines", name=f"{a}: largo plazo",
                             line=dict(color=BBVA["gris_oscuro"], dash="dash", width=1), showlegend=(i_ == 1)), row=i_, col=1)
fig.update_yaxes(title="σ anualizada (%)")
mostrar(fig, "08_volatilidad_condicional", "Volatilidad condicional anualizada (%) del GARCH(1,1)", 700)

tabla_parametros = pd.concat(filas_par).set_index(["accion", "parametro"])
tabla_diagnostico = pd.DataFrame(filas_diag).set_index("accion")
print("Parámetros del modelo final (H0: coeficiente = 0):")
print(tabla_parametros.to_string(float_format=lambda x: f"{x:.4g}"))
print("\nDiagnóstico:"); print(tabla_diagnostico.to_string(float_format=lambda x: f"{x:.4g}"))

print("\nINTERPRETACIÓN")
for a in ACCIONES_CARTERA:
    d = tabla_diagnostico.loc[a]
    bN, bT = tabla_seleccion.loc[(a, "GARCH(1,1)-N"), "BIC"], tabla_seleccion.loc[(a, "GARCH(1,1)-t"), "BIC"]
    print(f"- {a}: el mejor BIC global es {mejor_bic[a]}. Con innovaciones t el BIC {si(bT < bN, 'baja', 'sube')} {abs(bN - bT):.1f} puntos "
          f"respecto a la normal" + (f" (ν≈{d['nu']:.1f}, colas pesadas)." if spec_final[a].endswith("t") else "."))
    print(f"  α={d['alpha']:.3f}, β={d['beta']:.3f}, α+β={d['alpha+beta']:.3f}: la persistencia es {si(d['alpha+beta'] > 0.95, 'muy alta', si(d['alpha+beta'] > 0.85, 'alta', 'moderada'))} "
          f"y un shock pierde la mitad de su efecto en unos {d['semivida_dias']:.0f} días.")
    print(f"  Coeficientes {si(d['coefs_significativos'], 'todos significativos al 5%', 'no todos significativos al 5% (ver tabla)')}. "
          f"Ljung-Box de residuos estandarizados p={d['LB_resid_p']:.3f} y de sus cuadrados p={d['LB_resid2_p']:.3f}: "
          + si(d['LB_resid_p'] >= ALPHA_TEST and d['LB_resid2_p'] >= ALPHA_TEST,
               "no queda autocorrelación ni efecto ARCH sin explicar.",
               "queda dependencia sin recoger, lo anoto como limitación."))

Parámetros del modelo final (H0: coeficiente = 0):
                    coef  std_err     t   p_valor
accion parametro                                 
WMT    mu        0.08465  0.02262 3.743 0.0001817
       omega      0.1592  0.06096 2.611  0.009023
       alpha[1]   0.1191  0.03696 3.222  0.001273
       beta[1]    0.7887  0.06274 12.57         0
       nu          3.962   0.4091 9.685         0
MRK    mu         0.0725  0.02616 2.771  0.005586
       omega      0.1024  0.04488  2.28   0.02258
       alpha[1]  0.06164  0.01896 3.251   0.00115
       beta[1]    0.8944  0.03535  25.3         0
       nu          3.858   0.3558 10.84         0
NVDA   mu         0.2584  0.05757 4.488 7.186e-06
       omega       0.314   0.1087 2.887  0.003884
       alpha[1]  0.09665  0.01886 5.126 2.964e-07
       beta[1]    0.8773  0.02331 37.63         0
       nu          5.778    0.823 7.021 2.199e-12

Diagnóstico:
         alpha   beta  alpha+beta  semivida_dias  vol_LP_diaria_%    nu  LB_resid_p  

## 5 bis. Validación de mi implementación de GARCH
Como la estimación es propia (máxima verosimilitud con `scipy`), la contrasto con la librería `arch` sobre datos simulados con parámetros conocidos. Si coinciden los parámetros y la verosimilitud, la implementación es fiable. (Esta celda solo necesita `arch` instalada, `pip install arch`; no se usa para el resto del análisis.)

In [13]:
def simular_garch(n, mu, om, al, be, nu, seed=0, burn=500):
    rng = np.random.default_rng(seed)
    z = rng.standard_t(nu, n + burn) * np.sqrt((nu - 2) / nu)
    s2 = np.empty(n + burn); e = np.empty(n + burn); s2[0] = om / (1 - al - be)
    for t_ in range(n + burn):
        if t_ > 0: s2[t_] = om + al * e[t_ - 1] ** 2 + be * s2[t_ - 1]
        e[t_] = np.sqrt(s2[t_]) * z[t_]
    return mu + e[burn:]

verdad = {"mu": 0.05, "omega": 0.05, "alpha[1]": 0.08, "beta[1]": 0.90, "nu": 6.0}
y_sim = simular_garch(3000, verdad["mu"], verdad["omega"], verdad["alpha[1]"], verdad["beta[1]"], verdad["nu"])
mio = ajustar(y_sim, 1, 1, "t")
comp = pd.DataFrame({"valor_real": [verdad[n] for n in mio["nombres"]], "mi_implementacion": mio["theta"]}, index=mio["nombres"])
try:
    from arch import arch_model
    res_arch = arch_model(y_sim, mean="Constant", vol="GARCH", p=1, q=1, dist="t").fit(disp="off")
    comp["arch"] = res_arch.params[mio["nombres"]].to_numpy()
    print(comp.round(4).to_string())
    print(f"\nLog-verosimilitud: mía {mio['loglik']:.3f} | arch {res_arch.loglikelihood:.3f}")
    print("Máxima diferencia absoluta de parámetros frente a arch:", round(float(np.abs(comp["mi_implementacion"] - comp["arch"]).max()), 5))
except ImportError:
    print(comp.round(4).to_string()); print("\n`arch` no está instalada: pip install arch para completar la comparación.")

          valor_real  mi_implementacion    arch
mu              0.05             0.0736  0.0738
omega           0.05             0.0518  0.0521
alpha[1]        0.08             0.0771  0.0767
beta[1]         0.90             0.9004  0.9005
nu              6.00             5.7981  5.8276

Log-verosimilitud: mía -5073.206 | arch -5072.187
Máxima diferencia absoluta de parámetros frente a arch: 0.02947


## 6. Predicción de la volatilidad fuera de muestra
Evalúo la predicción con una **ventana expansiva**: reestimo el GARCH(1,1) cada `REFIT_CADA` sesiones con todos los datos anteriores y predigo la varianza un paso por delante. Entre reestimaciones mantengo los parámetros y actualizo la varianza con el retorno observado. Las predicciones solo usan información anterior al día predicho.

Comparo el GARCH(1,1) con una referencia de **volatilidad constante** (desviación típica de todo el histórico anterior). Las métricas son:
- **MAE** de la volatilidad predicha (σ) frente al valor absoluto del retorno (|r|).
- **MSE** de la varianza predicha (σ²) frente al retorno al cuadrado (r²).

|r| y r² son aproximaciones ruidosas de la volatilidad real, que no se observa. Por tanto, interpreto estas métricas como indicadores orientativos y no como una medida perfecta de la volatilidad verdadera.

In [14]:
def parametros_garch11(m):
    d = dict(zip(m["nombres"], m["theta"]))
    return d["mu"], d["omega"], d["alpha[1]"], d["beta[1]"]

if not (1 <= N_TEST < len(retornos) - 500):
    raise ValueError("N_TEST debe dejar al menos 500 observaciones de entrenamiento.")

pred, filas_m = {}, []
for a in ACCIONES_CARTERA:
    y = retornos[a].to_numpy() * 100; n = len(y); ini = n - N_TEST
    dist = "t" if spec_final[a].endswith("t") else "normal"
    v_g, v_c = np.empty(N_TEST), np.empty(N_TEST)

    u_prev = None
    for s in range(ini, n, REFIT_CADA):
        fin = min(s + REFIT_CADA, n)
        m = ajustar(y[:s], 1, 1, dist, inicio=u_prev); u_prev = m["u"]
        mu, om, al, be = parametros_garch11(m)
        s2 = om + al * (y[s - 1] - mu) ** 2 + be * m["sigma2"][-1]
        var_const = np.var(y[:s], ddof=1)
        for t_ in range(s, fin):
            v_g[t_ - ini], v_c[t_ - ini] = s2, var_const
            s2 = om + al * (y[t_] - mu) ** 2 + be * s2

    obs = y[ini:]; r2 = obs ** 2; absr = np.abs(obs)
    pred[a] = {"fechas": retornos.index[ini:], "abs_r": absr,
               "GARCH(1,1)": np.sqrt(v_g), "Constante": np.sqrt(v_c)}
    for nombre, v in (("GARCH(1,1)", v_g), ("Constante", v_c)):
        filas_m.append({"accion": a, "modelo": nombre,
                        "MAE_sigma_vs_|r|": np.mean(np.abs(np.sqrt(v) - absr)),
                        "MSE_var_vs_r2": np.mean((v - r2) ** 2)})

tabla_prediccion = pd.DataFrame(filas_m).set_index(["accion", "modelo"])
print(f"Evaluación fuera de muestra: últimas {N_TEST} sesiones, reestimación cada {REFIT_CADA}.")
print(tabla_prediccion.to_string(float_format=lambda x: f"{x:.5g}"))

fig = make_subplots(rows=len(ACCIONES_CARTERA), cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=[f"{a}: volatilidad diaria predicha (%)" for a in ACCIONES_CARTERA])
for i_, a in enumerate(ACCIONES_CARTERA, 1):
    p = pred[a]; lg = (i_ == 1)
    fig.add_trace(go.Scatter(x=p["fechas"], y=p["abs_r"], name="|retorno| (proxy)", mode="lines", line=dict(color=BBVA["gris_claro"], width=1), showlegend=lg), row=i_, col=1)
    fig.add_trace(go.Scatter(x=p["fechas"], y=p["GARCH(1,1)"], name="GARCH(1,1)", mode="lines", line=dict(color=COLORES[a], width=2), showlegend=lg), row=i_, col=1)
    fig.add_trace(go.Scatter(x=p["fechas"], y=p["Constante"], name="Constante", mode="lines", line=dict(color=BBVA["gris_oscuro"], width=1, dash="dash"), showlegend=lg), row=i_, col=1)
mostrar(fig, "09_prediccion_volatilidad", "Predicción de volatilidad fuera de muestra", 720)

print("\nINTERPRETACIÓN")
for a in ACCIONES_CARTERA:
    m = tabla_prediccion.loc[a]
    mejor_mae = m["MAE_sigma_vs_|r|"].idxmin()
    mejor_mse = m["MSE_var_vs_r2"].idxmin()
    print(f"- {a}: menor MAE de volatilidad con {mejor_mae}; menor MSE de varianza con {mejor_mse}.")
    print("  Estas métricas son orientativas porque |r| y r² son proxies ruidosos de la volatilidad no observada.")

Evaluación fuera de muestra: últimas 750 sesiones, reestimación cada 20.
                   MAE_sigma_vs_|r|  MSE_var_vs_r2
accion modelo                                     
WMT    GARCH(1,1)            0.7086         30.127
       Constante             0.8281         30.095
MRK    GARCH(1,1)           0.88406         45.733
       Constante            0.88791         45.647
NVDA   GARCH(1,1)             1.767         851.88
       Constante              1.909         837.71



INTERPRETACIÓN
- WMT: menor MAE de volatilidad con GARCH(1,1); menor MSE de varianza con Constante.
  Estas métricas son orientativas porque |r| y r² son proxies ruidosos de la volatilidad no observada.
- MRK: menor MAE de volatilidad con GARCH(1,1); menor MSE de varianza con Constante.
  Estas métricas son orientativas porque |r| y r² son proxies ruidosos de la volatilidad no observada.
- NVDA: menor MAE de volatilidad con GARCH(1,1); menor MSE de varianza con Constante.
  Estas métricas son orientativas porque |r| y r² son proxies ruidosos de la volatilidad no observada.


## 7. Volatilidad y correlaciones para el componente de Monte Carlo
Calculo tres estimaciones de la volatilidad anual de cada activo:
- **Histórica**: desviación típica de los retornos logarítmicos × √252.
- **GARCH de largo plazo**: `√(ω/(1−α−β)) × √252`.
- **GARCH a horizonte**: raíz de la media de la varianza que predice el GARCH durante los 5 años de vida de la opción (1260 sesiones). Es la volatilidad constante equivalente a la dinámica del modelo en ese horizonte y es la que uso por defecto (`FUENTE_SIGMA_MC`).

En cada uno de los 20 pasos de la simulación, la volatilidad que multiplica al movimiento browniano es `σ_paso = σ_anual·√Δt`, con Δt = 0.25 años. No se debe usar la σ diaria con ese Δt.

La media histórica de los retornos **no** se utiliza: en la valoración riesgo-neutral todos los activos crecen a la tasa libre de riesgo (μ = 0.01).

In [15]:
def varianza_horizonte(m, y, H):
    """Varianza GARCH(1,1) prevista de t+1 a t+H: LP + (α+β)^(h-1)·(σ²_{t+1} − LP)."""
    mu, om, al, be = parametros_garch11(m)
    s1 = om + al * (y[-1] - mu) ** 2 + be * m["sigma2"][-1]
    lp = om / (1 - al - be)
    return lp + (al + be) ** np.arange(H) * (s1 - lp)

H = TTM_ANIOS * DIAS_ANIO
filas = []
for a in ACCIONES_CARTERA:
    m = modelos_final[a]; y = retornos[a].to_numpy() * 100
    mu, om, al, be = parametros_garch11(m)
    sig_hist = float(retornos[a].std(ddof=1))
    sig_lp = float(np.sqrt(om / (1 - al - be)) / 100)
    sig_hor = float(np.sqrt(varianza_horizonte(m, y, H).mean()) / 100)
    sig_act = float(np.sqrt(m["sigma2"][-1]) / 100)
    anual = {"historica": sig_hist, "garch_largo_plazo": sig_lp, "garch_horizonte": sig_hor}
    s_mc = anual[FUENTE_SIGMA_MC] * np.sqrt(DIAS_ANIO)
    filas.append({"accion": a,
                  "sigma_diaria_historica": sig_hist, "sigma_diaria_garch_LP": sig_lp,
                  "sigma_diaria_garch_horizonte": sig_hor, "sigma_diaria_condicional_actual": sig_act,
                  "sigma_anual_historica": sig_hist * np.sqrt(DIAS_ANIO),
                  "sigma_anual_garch_LP": sig_lp * np.sqrt(DIAS_ANIO),
                  "sigma_anual_garch_horizonte": sig_hor * np.sqrt(DIAS_ANIO),
                  "sigma_anual_MC": s_mc, "sigma_paso_MC": s_mc * np.sqrt(DT_PASO)})
tabla_volatilidad = pd.DataFrame(filas).set_index("accion").loc[ACCIONES_CARTERA]
print(f"Fuente de σ para Monte Carlo: {FUENTE_SIGMA_MC} | Δt = {DT_PASO} años")
print(tabla_volatilidad.T.to_string(float_format=lambda x: f"{x:.6f}"))

rho_mc = rho.loc[ACCIONES_CARTERA, ACCIONES_CARTERA]
L = np.linalg.cholesky(rho_mc.to_numpy())          # falla si rho no es definida positiva
assert np.allclose(L @ L.T, rho_mc.to_numpy())
print("\nMatriz de correlaciones (retornos logarítmicos); Cholesky correcto, número de condición:",
      round(float(np.linalg.cond(rho_mc)), 2))
print(rho_mc.round(4))

RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
tabla_volatilidad.to_csv(RUTA_SALIDA); rho_mc.to_csv(RUTA_RHO)
print(f"\nGuardado: {RUTA_SALIDA} y {RUTA_RHO}")

fig = go.Figure()
for (c, lab, col) in (("sigma_anual_historica", "Histórica", BBVA["azul_muy_claro"]), ("sigma_anual_garch_LP", "GARCH largo plazo", BBVA["azul_claro"]),
                      ("sigma_anual_garch_horizonte", "GARCH horizonte 5 años", BBVA["azul_principal"])):
    fig.add_trace(go.Bar(x=ACCIONES_CARTERA, y=tabla_volatilidad[c] * 100, name=lab, marker_color=col, text=(tabla_volatilidad[c] * 100).round(1), textposition="outside"))
fig.update_layout(barmode="group", hovermode="closest"); fig.update_yaxes(title="σ anual (%)")
mostrar(fig, "10_volatilidad_por_metodo", "Volatilidad anual según el método", 430)

fig = go.Figure(go.Heatmap(z=rho_mc.to_numpy(), x=ACCIONES_CARTERA, y=ACCIONES_CARTERA, zmin=0, zmax=1, text=rho_mc.round(3).to_numpy(), texttemplate="%{text}",
                           colorscale=[[0, BBVA["blanco"]], [0.5, BBVA["azul_muy_claro"]], [1, BBVA["azul_principal"]]], colorbar=dict(title="ρ")))
fig.update_yaxes(autorange="reversed"); fig.update_layout(hovermode="closest")
mostrar(fig, "11_matriz_correlaciones", "Matriz de correlaciones ρ para Monte Carlo", 420)

print("\nINTERPRETACIÓN")
dif = (tabla_volatilidad["sigma_anual_garch_horizonte"] / tabla_volatilidad["sigma_anual_historica"] - 1) * 100
for a in ACCIONES_CARTERA:
    t = tabla_volatilidad.loc[a]
    print(f"- {a}: σ anual histórica {t['sigma_anual_historica']*100:.1f}%, GARCH a horizonte {t['sigma_anual_garch_horizonte']*100:.1f}% "
          f"({dif[a]:+.1f}%); σ condicional actual anualizada {t['sigma_diaria_condicional_actual']*np.sqrt(DIAS_ANIO)*100:.1f}%. "
          f"Uso {t['sigma_anual_MC']*100:.1f}% anual, es decir {t['sigma_paso_MC']*100:.2f}% por paso de {DT_PASO} años.")
print("Las diferencias entre métodos son " + si(dif.abs().max() < 10, "pequeñas (menos del 10%), así que el resultado de Monte Carlo no debería depender mucho de la elección.",
      "apreciables (más del 10% en algún activo), por lo que conviene analizar la sensibilidad de la prima a σ."))

Fuente de σ para Monte Carlo: garch_horizonte | Δt = 0.25 años
accion                               WMT      MRK     NVDA
sigma_diaria_historica          0.013805 0.014862 0.032594
sigma_diaria_garch_LP           0.013137 0.015256 0.034701
sigma_diaria_garch_horizonte    0.013118 0.015244 0.034396
sigma_diaria_condicional_actual 0.010989 0.014890 0.023236
sigma_anual_historica           0.219152 0.235933 0.517418
sigma_anual_garch_LP            0.208541 0.242184 0.550862
sigma_anual_garch_horizonte     0.208239 0.241997 0.546017
sigma_anual_MC                  0.208239 0.241997 0.546017
sigma_paso_MC                   0.104120 0.120998 0.273009

Matriz de correlaciones (retornos logarítmicos); Cholesky correcto, número de condición: 1.91
symbol     WMT     MRK    NVDA
symbol                        
WMT     1.0000  0.2345  0.2255
MRK     0.2345  1.0000  0.1295
NVDA    0.2255  0.1295  1.0000

Guardado: Datos\volatilidad_cartera.csv y Datos\correlaciones_cartera.csv



INTERPRETACIÓN
- WMT: σ anual histórica 21.9%, GARCH a horizonte 20.8% (-5.0%); σ condicional actual anualizada 17.4%. Uso 20.8% anual, es decir 10.41% por paso de 0.25 años.
- MRK: σ anual histórica 23.6%, GARCH a horizonte 24.2% (+2.6%); σ condicional actual anualizada 23.6%. Uso 24.2% anual, es decir 12.10% por paso de 0.25 años.
- NVDA: σ anual histórica 51.7%, GARCH a horizonte 54.6% (+5.5%); σ condicional actual anualizada 36.9%. Uso 54.6% anual, es decir 27.30% por paso de 0.25 años.
Las diferencias entre métodos son pequeñas (menos del 10%), así que el resultado de Monte Carlo no debería depender mucho de la elección.


## 8. Resumen y limitaciones

In [16]:
print("=" * 78); print("RESUMEN"); print("=" * 78)
for a in ACCIONES_CARTERA:
    d = tabla_diagnostico.loc[a]; m = tabla_prediccion.loc[a]
    mae = m["MAE_sigma_vs_|r|"].to_dict(); mse = m["MSE_var_vs_r2"].to_dict()
    print(f"{a}: serie de retornos {tabla_estacionariedad.loc[(a, 'retorno log (c)'), 'conclusion'].lower()}; "
          f"ARCH {si(tabla_arch.loc[a, 'ARCH_p'] < ALPHA_TEST, 'sí', 'no')}; modelo {spec_final[a]}; "
          f"α+β={d['alpha+beta']:.3f}; MAE GARCH/constante={mae['GARCH(1,1)']:.3f}/{mae['Constante']:.3f}; "
          f"MSE GARCH/constante={mse['GARCH(1,1)']:.3f}/{mse['Constante']:.3f}")
print("\nσ anual para Monte Carlo (%):", (tabla_volatilidad["sigma_anual_MC"] * 100).round(2).to_dict())

RESUMEN
WMT: serie de retornos estacionaria; ARCH sí; modelo GARCH(1,1)-t; α+β=0.908; MAE GARCH/constante=0.709/0.828; MSE GARCH/constante=30.127/30.095
MRK: serie de retornos estacionaria; ARCH sí; modelo GARCH(1,1)-t; α+β=0.956; MAE GARCH/constante=0.884/0.888; MSE GARCH/constante=45.733/45.647
NVDA: serie de retornos estacionaria; ARCH sí; modelo GARCH(1,1)-t; α+β=0.974; MAE GARCH/constante=1.767/1.909; MSE GARCH/constante=851.883/837.711

σ anual para Monte Carlo (%): {'WMT': 20.82, 'MRK': 24.2, 'NVDA': 54.6}


**Limitaciones que tengo en cuenta**
- Modelizo cada activo por separado con un GARCH univariante y asumo correlaciones constantes (la matriz ρ es la de toda la muestra). En crisis las correlaciones suelen subir.
- En Monte Carlo la volatilidad es constante, mientras que el GARCH describe una volatilidad que cambia. Por eso uso una σ equivalente para los 5 años; la simulación no recoge los agrupamientos de volatilidad.
- GARCH(1,1) es simétrico: no distingue entre bajadas y subidas. Un modelo asimétrico (GJR, EGARCH) podría capturar el efecto apalancamiento.
- La evaluación fuera de muestra usa |r| y r² como aproximación de la volatilidad real, y una ventana de test de tres años, así que sus conclusiones son orientativas.
- La media se modela como constante; las autocorrelaciones de los retornos son pequeñas, pero no nulas en todos los casos.

In [17]:
# Índice HTML con todos los gráficos exportados (entregable: html y js de los gráficos)
items = "\n".join(f'<h3 style="color:#001391;font-family:Arial">{n}</h3><iframe src="{n}.html" width="100%" height="780" style="border:1px solid #E4E4EC"></iframe>' for n in GRAFICOS)
(RUTA_GRAFICOS / "index.html").write_text(f'<html><head><meta charset="utf-8"><title>Gráficos Reto 9</title></head><body style="margin:24px;background:#fff">{items}</body></html>', encoding="utf-8")
print(f"{len(GRAFICOS)} gráficos exportados en {RUTA_GRAFICOS}/ (abrir index.html)")

17 gráficos exportados en figuras/ (abrir index.html)
